# ColliderML Release 1 — clustered view explorer

Reads the MLPF-converted parquet written by `mlpf.data.colliderml.postprocessing` and shows:

1. **Input/target distributions** of tracks, clusters, hits.
2. **Interactive 3D event display** — tracks, cluster spheres, calo hits, and tracker hits
   as toggleable layers in one figure.

Point `DATA_DIR` at any directory containing `train-*-of-*.parquet` in the MLPF schema.
The production output lives under `/mnt/ceph/users/ewulff/data/colliderml/mlpf_parquet/clustered/ttbar_pu0/`
(shards are organised per sample; `ttbar_pu0_old/` holds the pre-2026-09-21 conversion).

In [ ]:
import os
from pathlib import Path

import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# point at any directory of converted MLPF parquet shards
DATA_DIR = Path(os.environ.get("COLLIDERML_PARQUET", "/mnt/ceph/users/ewulff/data/colliderml/mlpf_parquet/clustered/ttbar_pu0"))

files = sorted(DATA_DIR.glob("train-*-of-*.parquet"))
print(f"{len(files)} shard(s) found under {DATA_DIR}")
for i, f in enumerate(files):
    print(f' {i}: {f.name}')
    if i > 4:
        print('  ...')
        break

In [ ]:
raw = ak.from_parquet(files[0])
FIELDS = list(raw.fields)
n_events = int(ak.num(raw["event_id"], axis=0))

# fixed per-event widths, only needed in the pathological case of an ENTIRE column being
# empty (flatten then loses the inner width — same parquet readback quirk the validator's
# H2 gate works around). A zero-row event inside a non-empty column needs no help: np.split
# on the (N, w) buffer yields (0, w) views automatically.
WIDTHS = {
    "X_track": 17, "X_cluster": 17,
    "ytarget_track": 14, "ytarget_cluster": 14,
    "X_hit_tracker": 12, "X_hit_calo": 12,
    "ytarget_hit_tracker": 14, "ytarget_hit_calo": 14,
}

# one plain value per event rather than a jagged row (the converter stacks them as scalars)
SCALAR_FIELDS = {"event_id", "genmet"}

def _flatten_field(field):
    """Flatten one jagged column into a single whole-shard numpy array."""
    col = raw[field]
    if field in SCALAR_FIELDS:
        return ak.to_numpy(col).reshape(-1, 1)
    flat_col = ak.to_numpy(ak.flatten(col, axis=1))
    if flat_col.size == 0 and field in WIDTHS:
        flat_col = flat_col.reshape(0, WIDTHS[field])
    return flat_col

# flat[field] = the whole shard as one (total_rows, w) array; per-event views are np.split
# slices of it at the event boundaries
flat = {f: _flatten_field(f) for f in FIELDS}
counts = {f: ak.to_numpy(ak.num(raw[f], axis=1)) for f in FIELDS if f not in SCALAR_FIELDS}
per_field = {
    f: (list(flat[f]) if f in SCALAR_FIELDS else np.split(flat[f], np.cumsum(counts[f])[:-1]))
    for f in FIELDS
}
events = [dict(zip(FIELDS, vals)) for vals in zip(*per_field.values())]
print(f"{len(events)} events loaded")
print(f"len(events): {len(events)}")
print("n_events:", n_events)
print(f"FIELDS: {FIELDS}")

In [ ]:
# pick the event we'll use for 3D displays and per-event inspection
IEV = 0
ev = events[IEV]  # dict of 2-D numpy arrays (zero-row events normalized to (0, w))

X_track = ev["X_track"]           # (n_track, 17)
X_cluster = ev["X_cluster"]       # (n_cluster, 17)
X_hit_trk = ev["X_hit_tracker"]   # (n_tracker, 12)
X_hit_cal = ev["X_hit_calo"]      # (n_calo, 12)
y_track = ev["ytarget_track"]     # (n_track, 14)
y_cluster = ev["ytarget_cluster"] # (n_cluster, 14)
y_hit_trk = ev["ytarget_hit_tracker"]
y_hit_cal = ev["ytarget_hit_calo"]

# no padding rows in this format: every row is a real element with nonzero elemtype
# in col 0 (1=track/tracker hit, 2=cluster/calo hit), so row counts are the element counts
print(f"event {IEV}: {len(X_track)} tracks, {len(X_cluster)} clusters, {len(X_hit_trk)} tracker hits, {len(X_hit_cal)} calo hits")

In [ ]:
X_track.shape

## 1. Input / target distributions per event
Aggregated over all events in the loaded shard. Inputs come from the X tables, targets from ytarget.

In [ ]:
# Whole-shard kinematics for the 1-D distributions: the flat buffers from the load cell,
# used directly — concatenating the per-event views would just rebuild a copy of these.
# Zero-row events contribute zero rows here, so no filtering is needed.
Xs_trk = flat["X_track"]
Xs_clu = flat["X_cluster"]
ys_trk = flat["ytarget_track"]
ys_clu = flat["ytarget_cluster"]
Xs_hcal = flat["X_hit_calo"]
Xs_htrk = flat["X_hit_tracker"]

# ytarget rows with PID>0 are the representatives (the parity of truth particles under the exclusive assignment)
m_y_trk = ys_trk[:, 0] > 0
m_y_clu = ys_clu[:, 0] > 0

print(f"inputs: {len(Xs_trk)} tracks, {len(Xs_clu)} clusters, {len(Xs_htrk)} tracker hits, {len(Xs_hcal)} calo hits")
print(f"targets: {m_y_trk.sum()} track representatives, {m_y_clu.sum()} cluster representatives")

In [ ]:
# X layout: tracks and clusters share the 17-wide EDM4hep joint layout. Tracks fill
# kinematics (cols 1..5) and perigee (cols 6..9 d0,z0,theta,qop); clusters fill kinematics plus
# centroid (6..8), energy split (10..12), n_hits (13), width sigma_x/y/z (14..16). Tracker hits
# only carry position (cols 6..8) and time (9); their "energy" and momentum columns are zeroed.
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

def _h(ax, data_a, data_b, name, la, lb, bins=60, logx=False, xmin=None):
    # a log x-axis needs log-spaced edges, shared by both calls so the steps stay
    # comparable: span xmin (falling back to the combined positive min) to the combined max
    if logx:
        lo = xmin if xmin is not None else min(data_a[data_a > 0].min(), data_b[data_b > 0].min())
        hi = max(data_a.max(), data_b.max())
        bin_edges = np.logspace(np.log10(lo), np.log10(hi), bins + 1)
    else:
        bin_edges = bins
    ax.hist(data_a, bins=bin_edges, histtype="step", color="#4c72b0", label=la)
    ax.hist(data_b, bins=bin_edges, histtype="step", color="#c44e52", label=lb)
    ax.set_xlabel(name)
    ax.set_yscale("log")
    if logx:
        ax.set_xscale("log")
    if xmin is not None:
        ax.set_xlim(left=xmin)
    ax.legend(fontsize=8)

_h(axes[0, 0], Xs_trk[:, 1], Xs_clu[:, 1], "pt [GeV]", "tracks", "clusters", logx=True)
_h(axes[0, 1], Xs_trk[:, 2], Xs_clu[:, 2], "eta", "tracks", "clusters")
_h(axes[0, 2], Xs_trk[:, 5], Xs_clu[:, 5], "energy [GeV]", "tracks", "clusters", logx=True)

_h(axes[1, 0], Xs_htrk[:, 6], Xs_hcal[:, 6], "hit x [mm]", "tracker hits", "calo hits")
_h(axes[1, 1], Xs_htrk[:, 7], Xs_hcal[:, 7], "hit y [mm]", "tracker hits", "calo hits")
_h(axes[1, 2], Xs_htrk[:, 8], Xs_hcal[:, 8], "hit z [mm]", "tracker hits", "calo hits")

fig.suptitle("Input features — tracks, clusters, hits (all events)")
fig.tight_layout()
plt.show()

# calo-hit energy on its own (tracker-hit kinematics are zeroed by the converter, so a
# combined histogram would just be a spike at zero)
fig, ax = plt.subplots(figsize=(6, 4))
he = Xs_hcal[:, 5]
ax.hist(he, bins=np.logspace(np.log10(he[he > 0].min()), np.log10(he.max()), 81), color="#c44e52")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("calo hit energy [GeV]"); ax.set_title("Calo hit energy")
plt.show()

In [ ]:
# Target distributions: ytarget rows in canonical particle_feature_order:
#   0 PID (PDG-style values: 211=charged had, 130=neutral had, 22=photon, 11=e, 13=mu)
#   1 charge, 2 pt, 3 eta, 4 sin_phi, 5 cos_phi, 6 energy, 7 ispu, 8 generatorStatus,
#   9 simulatorStatus, 10 gp_to_track, 11 gp_to_cluster, 12 jet_idx, 13 particle_number.
# Only rows with PID > 0 are representatives; the rest are placeholders (PID=0 links the
# track/cluster to a truth particle via particle_number without duplicating the kinematics).
PID_NAME = {211: "ch. had", 130: "n. had", 22: "photon", 11: "e", 13: "mu"}

def _cands(y_trk, y_clu, col):
    return np.concatenate([y_trk[m_y_trk, col], y_clu[m_y_clu, col]])

def _logbins(d, lo=1.0, n=60):
    # log-spaced edges for the log x-axes of the kinematics panels; floor at lo GeV
    return np.logspace(np.log10(lo), np.log10(d.max()), n + 1)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

pid = _cands(ys_trk, ys_clu, 0).astype(int)
uniq = sorted(np.unique(pid))
lbl = [f"{PID_NAME.get(u, u)} ({int((pid == u).sum())})" for u in uniq]
axes[0, 0].bar(range(len(uniq)), [int((pid == u).sum()) for u in uniq], color="#4c72b0")
axes[0, 0].set_xticks(range(len(uniq))); axes[0, 0].set_xticklabels(lbl, rotation=20, ha="right", fontsize=8)
axes[0, 0].set_ylabel("targets"); axes[0, 0].set_title("PID of representatives")

axes[0, 1].hist(_cands(ys_trk, ys_clu, 1), bins=[-1.5, -0.5, 0.5, 1.5], color="#c44e52")
axes[0, 1].set_xticks([-1, 0, 1]); axes[0, 1].set_xlabel("charge"); axes[0, 1].set_title("Charge")

pt_rep = _cands(ys_trk, ys_clu, 2)
axes[0, 2].hist(pt_rep, bins=_logbins(pt_rep), color="#55a868")
axes[0, 2].set_xscale("log"); axes[0, 2].set_xlim(left=1.0); axes[0, 2].set_xlabel("pt [GeV]"); axes[0, 2].set_title("Transverse momentum")

axes[1, 0].hist(_cands(ys_trk, ys_clu, 3), bins=60, color="#8172b2")
axes[1, 0].set_xlabel("eta"); axes[1, 0].set_title("Pseudorapidity")

e_rep = _cands(ys_trk, ys_clu, 6)
axes[1, 1].hist(e_rep, bins=_logbins(e_rep), color="#937860")
axes[1, 1].set_xscale("log"); axes[1, 1].set_xlim(left=1.0); axes[1, 1].set_xlabel("energy [GeV]"); axes[1, 1].set_title("Energy")

axes[1, 2].hist(_cands(ys_trk, ys_clu, 7), bins=[-0.5, 0.5, 1.5], color="#da8bc3")
axes[1, 2].set_xticks([0, 1]); axes[1, 2].set_xlabel("ispu"); axes[1, 2].set_title("Pileup flag")

fig.suptitle("Target representatives (ytarget; gen-level reference is in section 1b)")
fig.tight_layout()
plt.show()

In [ ]:
# Exclusive-deposit audit: gp_to_cluster (col 11) is the calibrated energy the representative owns
# in its cluster(s). GP_TO_CLUSTER vs truth energy should scatter near the diagonal; outliers show
# fragmentation or unclustered deposit.
gpc = ys_clu[m_y_clu, 11]
egen = ys_clu[m_y_clu, 6]
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(egen, gpc + 1e-6, s=1, alpha=0.4)
ax.set_xscale("log"); ax.set_yscale("log")
ax.plot([1e-2, 1e3], [1e-2, 1e3], "k--", lw=0.8)
ax.set_xlabel("truth E [GeV]"); ax.set_ylabel("gp_to_cluster [GeV]")
ax.set_title("Exclusive cluster deposit vs truth energy")
plt.show()
print(f"median gp_to_cluster/E = {np.median(gpc / egen):.3f}  (R2 gate passes in [0.3, 1.1])")

## 1b. Truth (gen) vs target

Since 2026-09-21 the gen and target tiers are distinct: `genjet`/`genmet` describe the
pre-visibility *measurable* truth set (any attributed calibrated deposit or track hit share,
minus neutrinos; `truth.py` `genref_features`), while `targetjet`/`ytarget` hold the exclusive
target representatives. The overlays below compare the two jet collections, the per-event
energy closure, and the target/gen jet pT response for ΔR < 0.1-matched jets (the validator's
R1 metric). On `ttbar_pu0_old` shards (gen≡target format) the overlays collapse onto each
other by construction.

In [ ]:
# Truth vs target aggregates. genjet/targetjet rows are (pt, eta, phi, E); genmet is one
# scalar per event (see the load cell: SCALAR_FIELDS).
gj_ev = [e["genjet"] for e in events]
tj_ev = [e["targetjet"] for e in events]
gm_ev = flat["genmet"].reshape(-1)

E_gj = np.array([j[:, 3].sum() for j in gj_ev])
E_tj = np.array([j[:, 3].sum() for j in tj_ev])
E_tgt = np.array([e["ytarget_track"][:, 6].sum() + e["ytarget_cluster"][:, 6].sum() for e in events])
# target MET from the representative rows (placeholder rows are all-zero)
met_tgt = np.array(
    [
        np.hypot(
            (e["ytarget_track"][:, 2] * e["ytarget_track"][:, 5]).sum() + (e["ytarget_cluster"][:, 2] * e["ytarget_cluster"][:, 5]).sum(),
            (e["ytarget_track"][:, 2] * e["ytarget_track"][:, 4]).sum() + (e["ytarget_cluster"][:, 2] * e["ytarget_cluster"][:, 4]).sum(),
        )
        for e in events
    ]
)

closure = (E_tgt - (E_gj + gm_ev)) / np.maximum(E_gj + gm_ev, 1e-9)

# target/gen jet response: each target jet matched to the nearest gen jet within deltaR < 0.1
resp = []
for gj, tj in zip(gj_ev, tj_ev):
    if len(gj) == 0 or len(tj) == 0:
        continue
    for t in tj:
        dphi = (gj[:, 2] - t[2] + np.pi) % (2 * np.pi) - np.pi
        dr = np.sqrt((gj[:, 1] - t[1]) ** 2 + dphi**2)
        k = int(np.argmin(dr))
        if dr[k] < 0.1:
            resp.append(t[0] / gj[k, 0])
resp = np.asarray(resp)
print(f"closure: median {np.median(closure):+.3f}  p90 |.| {np.percentile(np.abs(closure), 90):.3f}")
print(f"jet response: median {np.median(resp):.3f}  frac>1 {float(np.mean(resp > 1.0)):.3f}  n_matched {len(resp)}")

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

nbins = max(int(counts["genjet"].max()), int(counts["targetjet"].max())) + 2
axes[0, 0].hist(counts["genjet"], bins=np.arange(nbins) - 0.5, histtype="step", color="#4c72b0", label="gen")
axes[0, 0].hist(counts["targetjet"], bins=np.arange(nbins) - 0.5, histtype="step", color="#c44e52", label="target")
axes[0, 0].set_yscale("log")
axes[0, 0].set_xlabel("jets / event")
axes[0, 0].set_title("Jet multiplicity")
axes[0, 0].legend(fontsize=8)

for ax, col, name in ((axes[0, 1], 0, "pt [GeV]"), (axes[0, 2], 1, "eta"), (axes[1, 0], 3, "energy [GeV]")):
    ga, ta = flat["genjet"][:, col], flat["targetjet"][:, col]
    if col in (0, 3):
        lo = min(ga[ga > 0].min(), ta[ta > 0].min())
        hi = max(ga.max(), ta.max())
        bins = np.logspace(np.log10(lo), np.log10(hi), 61)
        ax.set_xscale("log")
    else:
        bins = 60
    ax.hist(ga, bins=bins, histtype="step", color="#4c72b0", label="gen")
    ax.hist(ta, bins=bins, histtype="step", color="#c44e52", label="target")
    ax.set_xlabel(name)
    ax.set_yscale("log")
    ax.legend(fontsize=8)

axes[1, 1].hist(closure, bins=101, color="#55a868")
axes[1, 1].axvline(float(np.median(closure)), color="k", ls="--", lw=1)
axes[1, 1].set_xlabel("(E(target) - (E(genjets) + genmet)) / (E(genjets) + genmet)")
axes[1, 1].set_title(f"closure: median {np.median(closure):+.3f}, p90 |.| {np.percentile(np.abs(closure), 90):.3f}")

axes[1, 2].hist(resp, bins=np.linspace(0.5, 1.5, 101), color="#8172b2")
axes[1, 2].axvline(1.0, color="k", ls="--", lw=1)
axes[1, 2].set_xlabel("target jet pt / gen jet pt (ΔR < 0.1)")
axes[1, 2].set_title(f"median {np.median(resp):.3f}, frac>1 {np.mean(resp > 1.0):.3f} (n={len(resp)})")

fig.suptitle("Truth (gen) vs target — jets and energy closure")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
met_pos = np.concatenate([gm_ev[gm_ev > 0], met_tgt[met_tgt > 0]])
mbins = np.logspace(np.log10(met_pos.min()), np.log10(max(gm_ev.max(), met_tgt.max())), 61)
axes[0].hist(gm_ev, bins=mbins, histtype="step", color="#4c72b0", label="genmet (measurable truth)")
axes[0].hist(met_tgt, bins=mbins, histtype="step", color="#c44e52", label="target MET")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("MET [GeV]")
axes[0].legend(fontsize=8)
axes[1].scatter(gm_ev, met_tgt, s=4, alpha=0.4)
lim = 1.05 * max(gm_ev.max(), met_tgt.max())
axes[1].plot([0, lim], [0, lim], "k--", lw=0.8)
axes[1].set_xlim(0, lim)
axes[1].set_ylim(0, lim)
axes[1].set_xlabel("genmet [GeV]")
axes[1].set_ylabel("target MET [GeV]")
axes[1].set_title(f"median genmet {np.median(gm_ev):.1f} GeV vs target MET {np.median(met_tgt):.1f} GeV")
fig.suptitle("Truth (gen) vs target — missing ET")
fig.tight_layout()
plt.show()

## 2. 3D event display

Tracks are drawn as straight segments from the perigee point (reconstructed from d0/z0/phi) out
to the ECAL inner face — the helix curvature at the cm scale is invisible next to a detector
that spans metres.

Each cluster is drawn as a **sphere**: centre = energy-weighted hit centroid, **size ∝ log₁₀(E)**
(5 MeV → 2 px, 10 GeV → 22 px; sizes are in screen pixels, not mm), colour = dominant deposit
region (ECAL-only / HCAL-only / boundary-mixed). Calo hits are markers coloured by log energy;
hover a cluster for its exact energy and RMS extent.

Most clusters really are tiny: median E ≈ 0.03 GeV, ~55% are single cells (sigma=0), and the
multi-hit ones span ~4–30 mm RMS. The energies cover ~4 decades, so a sqrt(E) size mapping
would leave ~96% of markers at the pixel floor — hence the log₁₀ mapping. The energetic core
of the event is the top few percent of clusters.

Plotly renders interactively (rotate/zoom, legend toggles per layer).

In [ ]:
import plotly.graph_objects as go

# Cluster colour comes from the ecal/hcal energy split in X (cols 10/11). The region bitmask
# computed by _cluster_region is not written into X_cluster, so the region label is reconstructed
# here from the dominant deposit region. A cluster is "mixed" if both sides exceed 25%.
# Colours are saturated and nearly opaque: most clusters are small, low-energy ones
# (median E ~ 0.03 GeV), so faint alphas make them disappear against the hits.
COLOR_ECAL = "rgba(0, 92, 171, 0.95)"
COLOR_HCAL = "rgba(217, 32, 23, 0.95)"
COLOR_MIXED = "rgba(255, 140, 0, 0.95)"

def cluster_color(e_ecal, e_hcal):
    """Dominant-region colour from ecal/hcal energy fractions."""
    tot = max(e_ecal + e_hcal, 1e-12)
    f_e = e_ecal / tot
    if f_e > 0.75:
        return COLOR_ECAL
    if f_e < 0.25:
        return COLOR_HCAL
    return COLOR_MIXED

def track_perigee_pt3(d0, z0, phi):
    """(d0, z0, phi) perigee -> 3D start point. d0 is signed 90 deg clockwise from phi."""
    x = -d0 * np.sin(phi)
    y = d0 * np.cos(phi)
    return np.array([x, y, z0], dtype=np.float64)

# ECAL inner face radius (~1.25 m in the OpenDataDetector geometry); track segments run from
# the perigee out to here, matching where the calorimeter data begins
TRACK_R_MAX = 1250.0

def track_line(d0, z0, pt, eta, sin_phi, cos_phi, r_max=TRACK_R_MAX):
    """Straight segment from the perigee point to the ECAL inner face."""
    phi = np.arctan2(sin_phi, cos_phi)
    tx, ty = np.cos(phi), np.sin(phi)
    tz = np.sinh(eta)
    p = np.array([tx, ty, tz]) / np.linalg.norm([tx, ty, tz])
    start = track_perigee_pt3(d0, z0, phi)
    end = start + p * r_max
    return start, end

# Marker size encodes log10(E), not E or sqrt(E): cluster energies span ~4 decades
# (median ~0.03 GeV, 99th percentile ~4 GeV), so any linear/sqrt mapping leaves ~96%
# of clusters pinned at the pixel floor. [E_SIZE_LO, E_SIZE_HI] maps onto [MS_LO, MS_HI] px;
# above 10 GeV everything draws at full size.
E_SIZE_LO, E_SIZE_HI = 0.005, 10.0  # GeV
MS_LO, MS_HI = 2.0, 16.0            # px

def energy_marker_size(energies):
    """log10(E) -> marker diameter [px], clipped to [MS_LO, MS_HI]."""
    f = (np.log10(np.maximum(energies, 1e-12)) - np.log10(E_SIZE_LO)) / (np.log10(E_SIZE_HI) - np.log10(E_SIZE_LO))
    return np.clip(MS_LO + f * (MS_HI - MS_LO), MS_LO, MS_HI)

def spheres(centroids, energies, sigmas, colors, name):
    """One marker per cluster. Size ~ log10(E); energy and RMS extent go on the hover text.
    Sizing on sigma alone would hide the ~55% single-hit clusters (sigma=0) and blow up the
    widest ones."""
    ms = energy_marker_size(energies)
    return go.Scatter3d(
        x=centroids[:, 0], y=centroids[:, 1], z=centroids[:, 2],
        mode="markers",
        marker=dict(size=ms, color=colors, opacity=1.0, symbol="circle"),
        name=name,
        hovertemplate="%{text}<extra>%{fullData.name}</extra>",
        text=[f"E={e:.2f} GeV, sigma={s:.0f} mm" for e, s in zip(energies, sigmas)],
    )

def tracks_3d(d0, z0, pt, eta, sin_phi, cos_phi):
    """One Line3d per track; concatenating with None rows lets us use a single trace."""
    xs, ys, zs = [], [], []
    for i in range(len(d0)):
        s, e = track_line(d0[i], z0[i], pt[i], eta[i], sin_phi[i], cos_phi[i])
        xs += [s[0], e[0], None]
        ys += [s[1], e[1], None]
        zs += [s[2], e[2], None]
    return go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode="lines",
        line=dict(color="rgba(50, 50, 50, 0.7)", width=2),
        name="tracks",
        hoverinfo="skip",
    )

def calo_hits_3d(xyz, energies):
    """Raw calo hits, coloured by log10 energy; the colourbar title labels the scale."""
    return go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode="markers",
        marker=dict(size=2, color=np.log10(np.maximum(energies, 1e-9)), colorscale="Viridis",
                    colorbar=dict(title="log10 E [GeV]"), opacity=0.6),
        name="calo hits",
    )

def tracker_hits_3d(xyz):
    """Raw tracker hits — position only (the converter zeroes their kinematics until ACTS)."""
    return go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode="markers",
        marker=dict(size=1.5, color="rgba(50, 50, 200, 0.5)"),
        name="tracker hits",
    )

def scene_ranges_fixed(*xyz_arrays, pad_frac=0.02):
    """Per-axis [lo, hi] spanning ALL given (n, 3) layers. The scene axes are pinned to
    these (autorange=False), so plotly never re-fits the view to the *visible* traces when
    layers are toggled in the legend."""
    pts = np.concatenate([a for a in xyz_arrays if len(a)], axis=0)
    lo, hi = pts.min(axis=0), pts.max(axis=0)
    pad = np.maximum((hi - lo) * pad_frac, 1.0)
    return {ax: [float(l), float(h)] for ax, l, h in zip(("x", "y", "z"), lo - pad, hi + pad)}

In [ ]:
# Cluster geometry comes straight from X_cluster: cols 6..8 centroid, 5 energy, 10/11
# ecal/hcal energy, 14..16 sigma_x/y/z. The hit-to-cluster membership isn't in the parquet
# (the allocator keeps it internal), so the sphere rendering trusts X_cluster; the raw hits
# join as their own layers in the figure below.
cl_centroids = X_cluster[:, 6:9]
cl_energy = X_cluster[:, 5]
cl_ecal = X_cluster[:, 10]
cl_hcal = X_cluster[:, 11]
cl_sigma = (X_cluster[:, 14] + X_cluster[:, 15] + X_cluster[:, 16]) / 3.0
cl_colors = [cluster_color(e, h) for e, h in zip(cl_ecal, cl_hcal)]

# calo-hit positions + energy; tracker hits have position only
hcal_xyz = np.stack([X_hit_cal[:, 6], X_hit_cal[:, 7], X_hit_cal[:, 8]], axis=1)
hcal_e = X_hit_cal[:, 5]
htrk_xyz = np.stack([X_hit_trk[:, 6], X_hit_trk[:, 7], X_hit_trk[:, 8]], axis=1)

def base_layout(title, ranges=None):
    # start side-on so the beam axis (z) lies horizontal on screen: the camera sits in
    # the transverse plane (eye z=0, so physical z projects exactly horizontally) and
    # physical y is screen-up. Plotly's turntable rotation then keeps z horizontal
    # while dragging sideways.
    camera = dict(eye=dict(x=2.0, y=0.6, z=0.0), up=dict(x=0, y=1, z=0))
    # pin the legend (trace toggles) to the top-left: the default right-side placement
    # collides with the calo-hit colorbar
    legend = dict(x=0.01, y=0.99, xanchor="left", yanchor="top", bgcolor="rgba(255, 255, 255, 0.6)")
    def _axis(ax):
        d = dict(title=f"{ax} [mm]")
        if ranges is not None:
            d.update(range=ranges[ax], autorange=False)
        return d
    return dict(
        title=title,
        scene=dict(xaxis=_axis("x"), yaxis=_axis("y"), zaxis=_axis("z"), camera=camera),
        legend=legend,
        margin=dict(l=0, r=0, b=0, t=30),
        height=600,
    )

# track perigee params for the track lines
trk_d0 = X_track[:, 6]
trk_z0 = X_track[:, 7]
trk_pt = X_track[:, 1]
trk_eta = X_track[:, 2]
trk_sin = X_track[:, 3]
trk_cos = X_track[:, 4]

# fixed axis ranges spanning every layer (calo hits dominate the extent), so toggling
# layers in the legend never re-zooms the view; reused by the union_find figure below
if len(trk_d0):
    trk_phi = np.arctan2(trk_sin, trk_cos)
    trk_start = np.stack([-trk_d0 * np.sin(trk_phi), trk_d0 * np.cos(trk_phi), trk_z0], axis=1)
    trk_dir = np.stack([np.cos(trk_phi), np.sin(trk_phi), np.sinh(trk_eta)], axis=1)
    trk_dir /= np.linalg.norm(trk_dir, axis=1, keepdims=True)
    trk_end = trk_start + trk_dir * TRACK_R_MAX
else:
    trk_start = trk_end = np.zeros((0, 3))
ax_ranges = scene_ranges_fixed(hcal_xyz, htrk_xyz, cl_centroids, trk_start, trk_end)

# everything on one figure; layers toggle via the legend (top-left)
fig = go.Figure(
    data=[tracks_3d(trk_d0, trk_z0, trk_pt, trk_eta, trk_sin, trk_cos),
          spheres(cl_centroids, cl_energy, cl_sigma, cl_colors, "clusters"),
          calo_hits_3d(hcal_xyz, hcal_e),
          tracker_hits_3d(htrk_xyz)],
    layout=base_layout(f"ColliderML event {IEV} — tracks + clusters + hits", ax_ranges))
fig.show()


## Tips

- Bump `IEV` and re-run from the per-event cell to browse different events.
- Point `COLLIDERML_PARQUET` at any directory of converted shards, e.g. a smoke-test directory,
  via `os.environ["COLLIDERML_PARQUET"] = "/path"` before re-running the load cell.
- The hits-vs-clusters figure is the hit-based vs clustered comparison on the same event: raw
  calo hits, the clusterer's spheres, and tracks over both.
- Per-cluster hit membership is now saved in `hit_to_cluster` (ints, one per calo hit; -1 =
  unassigned). Colour each cluster sphere by that field and you get exact per-cluster contours
  rather than proxies; that's the V2-rule anatomy at a glance.